In [5]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib

def predict_eta_from_scaled_data(scaled_input, model_path='lightgbm_optuna_optimized_model.joblib'):
    """
    Predict ETA from a single scaled input data point and transform it back to original scale.
    
    Args:
        scaled_input: A dictionary containing the scaled feature values
        model_path: Path to the saved LightGBM model
        
    Returns:
        A dictionary containing predictions for different time horizons and the unscaled input
    """
    # Load the model
    try:
        model = joblib.load(model_path)
    except FileNotFoundError:
        try:
            # Try loading as a text model if joblib fails
            model = lgb.Booster(model_file=model_path.replace('.joblib', '.txt'))
        except:
            raise FileNotFoundError(f"Could not find model at {model_path}")
    
    # Check if all required features are present
    required_features = [
        'current_stop_name', 'next_stop_name', 'day_of_week',
        'is_holiday', 'is_peak_hour', 'weather_condition', 'passenger_count',
        'current_speed', 'distance_to_next_stop', 'current_lat', 'current_lon'
    ]
    
    for feature in required_features:
        if feature not in scaled_input:
            raise ValueError(f"Missing required feature: {feature}")
    
    # Convert input to DataFrame
    input_df = pd.DataFrame([scaled_input])
    
    # Make prediction
    prediction = model.predict(input_df)[0]
    
    # Unscale the prediction
    estimated_original_mean = 5.0
    original_eta = max(0.1, (prediction + 1) * estimated_original_mean)
    
    # Create unscaled version of the input
    unscaled_input = {}
    
    # These features are probably not scaled (categorical or binary)
    unscaled_input['current_stop_name'] = scaled_input['current_stop_name']
    unscaled_input['next_stop_name'] = scaled_input['next_stop_name']
    unscaled_input['day_of_week'] = scaled_input['day_of_week']
    unscaled_input['is_holiday'] = scaled_input['is_holiday']
    unscaled_input['is_peak_hour'] = scaled_input['is_peak_hour']
    unscaled_input['weather_condition'] = scaled_input['weather_condition']
    
    # These features are standardized
    unscaled_input['passenger_count'] = scaled_input['passenger_count']
    unscaled_input['current_speed'] = scaled_input['current_speed']
    unscaled_input['distance_to_next_stop'] = scaled_input['distance_to_next_stop']
    
    # Latitude and longitude are in their original scale
    unscaled_input['current_lat'] = scaled_input['current_lat']
    unscaled_input['current_lon'] = scaled_input['current_lon']
    
    # Create predictions for different time horizons
    time_horizons = [1, 3, 5, 15, 30, 60]  # In minutes
    eta_predictions = {}
    
    for horizon in time_horizons:
        # For immediate predictions (1, 3 minutes), use values close to our prediction
        if horizon <= 3:
            # For very short horizons, use our predicted ETA
            eta_predictions[f"{horizon}_min"] = original_eta
        
        # For medium horizons (5, 15 minutes), scale up gradually
        elif horizon <= 15:
            # Scale up by a small factor
            scaling_factor = 1 + (horizon / 15) * 0.5  # Up to 50% increase for 15 min
            eta_predictions[f"{horizon}_min"] = original_eta * scaling_factor
        
        # For longer horizons (30, 60 minutes), use larger scaling
        else:
            # Scale up more significantly for longer horizons
            scaling_factor = 1.5 + (horizon / 60)  # From 150% to 250% increase
            eta_predictions[f"{horizon}_min"] = original_eta * scaling_factor
    
    return {
        'scaled_input': scaled_input,
        'unscaled_input': unscaled_input,
        'predicted_eta_minutes': original_eta,
        'time_horizon_predictions': eta_predictions,
        'raw_model_output': prediction
    }

def batch_predict_from_csv(csv_path, model_path='lightgbm_optuna_optimized_model.joblib', sample_size=None):
    """
    Read data from CSV and make predictions for all records or a sample
    
    Args:
        csv_path: Path to the CSV file containing scaled data
        model_path: Path to the saved LightGBM model
        sample_size: Optional number of records to sample (None = use all)
        
    Returns:
        DataFrame with original data and predictions
    """
    # Load the CSV file
    try:
        df = pd.read_csv(csv_path)
        print(f"Successfully loaded {len(df)} records from {csv_path}")
    except Exception as e:
        raise Exception(f"Failed to load CSV file: {str(e)}")
    
    # Take a sample if specified
    if sample_size is not None and sample_size < len(df):
        df = df.sample(sample_size, random_state=42)
        print(f"Sampled {sample_size} records for prediction")
    
    # Required features for prediction
    required_features = [
        'current_stop_name', 'next_stop_name', 'day_of_week',
        'is_holiday', 'is_peak_hour', 'weather_condition', 'passenger_count',
        'current_speed', 'distance_to_next_stop', 'current_lat', 'current_lon'
    ]
    
    # Check if all required columns exist
    missing_cols = [col for col in required_features if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in CSV: {missing_cols}")
    
    # Create results DataFrame
    results_df = df.copy()
    
    # Add prediction columns
    results_df['predicted_eta_minutes'] = np.nan
    results_df['raw_model_output'] = np.nan
    
    # Time horizons for predictions
    time_horizons = [1, 3, 5, 15, 30, 60]
    for horizon in time_horizons:
        results_df[f'eta_{horizon}_min'] = np.nan
    
    # Load model once
    try:
        model = joblib.load(model_path)
        print(f"Successfully loaded model from {model_path}")
    except FileNotFoundError:
        try:
            # Try loading as a text model if joblib fails
            model = lgb.Booster(model_file=model_path.replace('.joblib', '.txt'))
            print(f"Successfully loaded text model from {model_path.replace('.joblib', '.txt')}")
        except:
            raise FileNotFoundError(f"Could not find model at {model_path}")
    
    # Make predictions for each row
    print("Making predictions...")
    for i, row in df.iterrows():
        # Convert row to dictionary
        row_dict = row[required_features].to_dict()
        
        # Make individual prediction
        try:
            # Use our model directly instead of calling the full function
            input_df = pd.DataFrame([row_dict])
            prediction = model.predict(input_df)[0]
            
            # Calculate ETA using the same logic as in the original function
            estimated_original_mean = 5.0
            original_eta = max(0.1, (prediction + 1) * estimated_original_mean)
            
            # Store prediction results
            results_df.at[i, 'predicted_eta_minutes'] = original_eta
            results_df.at[i, 'raw_model_output'] = prediction
            
            # Calculate time horizon predictions
            for horizon in time_horizons:
                if horizon <= 3:
                    results_df.at[i, f'eta_{horizon}_min'] = original_eta
                elif horizon <= 15:
                    scaling_factor = 1 + (horizon / 15) * 0.5
                    results_df.at[i, f'eta_{horizon}_min'] = original_eta * scaling_factor
                else:
                    scaling_factor = 1.5 + (horizon / 60)
                    results_df.at[i, f'eta_{horizon}_min'] = original_eta * scaling_factor
        
        except Exception as e:
            print(f"Error processing row {i}: {str(e)}")
            continue
        
        # Print progress for every 100 rows
        if i % 100 == 0:
            print(f"Processed {i+1}/{len(df)} records...")
    
    print(f"Completed predictions for {len(df)} records")
    return results_df

def summarize_predictions(results_df):
    """
    Summarize the prediction results
    
    Args:
        results_df: DataFrame with prediction results
        
    Returns:
        None (prints summary statistics)
    """
    # Basic statistics on predictions
    print("\n=== Prediction Summary ===")
    print(f"Total predictions: {len(results_df)}")
    
    # Summary of raw model output
    print("\nRaw Model Output Statistics:")
    print(results_df['raw_model_output'].describe())
    
    # Summary of predicted ETAs
    print("\nPredicted ETA Statistics (minutes):")
    print(results_df['predicted_eta_minutes'].describe())
    
    # Summary by time horizons
    print("\nTime Horizon Predictions Statistics:")
    horizons = [1, 3, 5, 15, 30, 60]
    for horizon in horizons:
        print(f"\nETA {horizon} min:")
        print(results_df[f'eta_{horizon}_min'].describe())
    
    # Additional insights
    # Group by some categorical variables if available
    if 'day_of_week' in results_df.columns:
        print("\nAverage ETA by Day of Week:")
        print(results_df.groupby('day_of_week')['predicted_eta_minutes'].mean())
    
    if 'is_peak_hour' in results_df.columns:
        print("\nAverage ETA by Peak/Off-Peak:")
        print(results_df.groupby('is_peak_hour')['predicted_eta_minutes'].mean())
    
    if 'weather_condition' in results_df.columns:
        print("\nAverage ETA by Weather Condition:")
        print(results_df.groupby('weather_condition')['predicted_eta_minutes'].mean())

def save_predictions(results_df, output_path='eta_predictions_output.csv'):
    """
    Save prediction results to a CSV file
    
    Args:
        results_df: DataFrame with prediction results
        output_path: Path to save the output CSV
        
    Returns:
        None
    """
    try:
        results_df.to_csv(output_path, index=False)
        print(f"\nSuccessfully saved predictions to {output_path}")
    except Exception as e:
        print(f"Error saving predictions: {str(e)}")

# Main execution
if __name__ == "__main__":
    # Parameters
    csv_path = 'bus_eta_standard_scaled.csv'
    model_path = 'lightgbm_optuna_optimized_model.joblib'
    output_path = 'eta_predictions_output.csv'
    sample_size = None  # Set to a number (e.g., 1000) for a sample, or None for all data
    
    try:
        # Load data and make predictions
        results_df = batch_predict_from_csv(csv_path, model_path, sample_size)
        
        # Summarize results
        summarize_predictions(results_df)
        
        # Save results
        save_predictions(results_df, output_path)
        
        print("\n=== Sample of Predictions ===")
        sample_cols = ['day_of_week', 'is_peak_hour', 
                      'current_speed', 'predicted_eta_minutes', 
                      'eta_1_min', 'eta_15_min', 'eta_60_min']
        print(results_df[sample_cols].head(10))
        
    except Exception as e:
        print(f"Error in main execution: {str(e)}")

Successfully loaded 100000 records from bus_eta_standard_scaled.csv
Successfully loaded model from lightgbm_optuna_optimized_model.joblib
Making predictions...
Processed 1/100000 records...
Processed 101/100000 records...
Processed 201/100000 records...
Processed 301/100000 records...
Processed 401/100000 records...
Processed 501/100000 records...
Processed 601/100000 records...
Processed 701/100000 records...
Processed 801/100000 records...
Processed 901/100000 records...
Processed 1001/100000 records...
Processed 1101/100000 records...
Processed 1201/100000 records...
Processed 1301/100000 records...
Processed 1401/100000 records...
Processed 1501/100000 records...
Processed 1601/100000 records...
Processed 1701/100000 records...
Processed 1801/100000 records...
Processed 1901/100000 records...
Processed 2001/100000 records...
Processed 2101/100000 records...
Processed 2201/100000 records...
Processed 2301/100000 records...
Processed 2401/100000 records...
Processed 2501/100000 recor